# Entendendo o Risco em Seguros de Automóveis
## Uma Análise Exploratória do Dataset Car Insurance Claims
## Por que estamos aqui?
Imagine que você é responsável por precificar seguros de automóveis. A pergunta que não sai da sua cabeça é: *"Como saber quanto cobrar de cada cliente?"*
Se você cobrar muito, perde clientes. Se cobrar pouco, perde dinheiro. A resposta está nos dados.
Nesta análise, vamos explorar um dataset com 10 mil clientes e descobrir padrões reais de risco. Veremos que alguns grupos têm 7 vezes mais chance de ter sinistro que outros. Essas diferenças são ouro puro para uma seguradora.
**Dataset**: Car Insurance Data (Kaggle)
**Objetivo**: Entender como a taxa de sinistro varia por segmento e identificar oportunidades de precificação melhor

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# No Kaggle, o dataset é automaticamente disponibilizado em /kaggle/input
# Você precisa adicionar o dataset como "Input" no Kaggle antes de executar
import os

# Listar o que está disponível
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if 'insurance' in file.lower() or 'csv' in file.lower():
            print(os.path.join(root, file))

# Carregar o CSV (ajuste o caminho conforme necessário)
df = pd.read_csv('/kaggle/input/datasets/sagnik1511/car-insurance-data/Car_Insurance_Claim.csv')

print(f'Dados carregados: {df.shape[0]:,} clientes, {df.shape[1]} informações por cliente')

/kaggle/input/datasets/sagnik1511/car-insurance-data/Car_Insurance_Claim.csv
Dados carregados: 10,000 clientes, 19 informações por cliente


## Primeira Olhada nos Dados
Quando você recebe um novo dataset, a primeira coisa é explorar. Que tipo de informação temos? Como os dados se parecem? Há problemas óbvios?
Vamos começar simples.

In [2]:
# Ver as primeiras linhas
print('Primeiros 5 clientes da base:')
print(df.head().to_string())
print(f'\n... e assim por diante, até {len(df):,} registros')

Primeiros 5 clientes da base:
       ID    AGE  GENDER      RACE DRIVING_EXPERIENCE    EDUCATION         INCOME  CREDIT_SCORE  VEHICLE_OWNERSHIP VEHICLE_YEAR  MARRIED  CHILDREN  POSTAL_CODE  ANNUAL_MILEAGE VEHICLE_TYPE  SPEEDING_VIOLATIONS  DUIS  PAST_ACCIDENTS  OUTCOME
0  569520    65+  female  majority               0-9y  high school    upper class      0.629027                1.0   after 2015      0.0       1.0        10238         12000.0        sedan                    0     0               0      0.0
1  750365  16-25    male  majority               0-9y         none        poverty      0.357757                0.0  before 2015      0.0       0.0        10238         16000.0        sedan                    0     0               0      1.0
2  199901  16-25  female  majority               0-9y  high school  working class      0.493146                1.0  before 2015      0.0       0.0        10238         11000.0        sedan                    0     0               0      0.0
3  478

In [3]:
# Entender os tipos de dados
print('\nQue tipo de informação temos?')
print('\nColunas numéricas (quantidades):')
print(df.select_dtypes(include=[np.number]).columns.tolist())
print(f'\nColunas categóricas (categorias):')
print(df.select_dtypes(include=['object']).columns.tolist())
print(f'\nTotal: {df.select_dtypes(include=[np.number]).shape[1]} numéricas + {df.select_dtypes(include=["object"]).shape[1]} categóricas')


Que tipo de informação temos?

Colunas numéricas (quantidades):
['ID', 'CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'MARRIED', 'CHILDREN', 'POSTAL_CODE', 'ANNUAL_MILEAGE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS', 'OUTCOME']

Colunas categóricas (categorias):
['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'VEHICLE_YEAR', 'VEHICLE_TYPE']

Total: 11 numéricas + 8 categóricas


## Limpeza de Dados: O Trabalho Chato (Mas Necessário)
Antes de confiar em qualquer análise, precisamos verificar a qualidade. Dados sujos = conclusões erradas.
Vamos procurar por:
- **Valores faltantes** (buracos nos dados)
- **Duplicatas** (mesma pessoa registrada 2x?)
- **Outliers** (valores muito estranhos)
- **Desbalanceamento** (temos sinistros de todos os tipos?)

In [4]:
# Verificar valores faltantes
missing_count = df.isnull().sum()
missing_df = missing_count[missing_count > 0].to_frame('count')
missing_df['percentage'] = (missing_df['count'] / len(df) * 100).round(2)

if len(missing_df) > 0:
    print('Valores faltantes encontrados:')
    print(missing_df.to_string())
    print(f'\nTotal: {missing_df["count"].sum()} valores faltantes em {len(df):,} registros')
else:
    print('Sem valores faltantes.')

Valores faltantes encontrados:
                count  percentage
CREDIT_SCORE      982        9.82
ANNUAL_MILEAGE    957        9.57

Total: 1939 valores faltantes em 10,000 registros


In [5]:
# Duplicatas
duplicates = df.duplicated().sum()
print(f'\nDuplicatas: {duplicates} registros')
if duplicates == 0:
    print(' Sem duplicatas - dados limpos nesse aspecto')
else:
    print(f'  {duplicates} registros aparecem mais de uma vez')


Duplicatas: 0 registros
 Sem duplicatas - dados limpos nesse aspecto


## Análise de Outliers

Outliers podem distorcer nossas análises. Vamos usar o método IQR (Interquartile Range) para identificá-los.
Este método é robusto e não assume distribuição normal.

In [ ]:
import numpy as np

numeric_cols = df.select_dtypes(include=[np.number]).columns
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_pct = len(outliers) / len(df) * 100
    
    outlier_summary[col] = {
        'count': len(outliers),
        'percentage': outlier_pct,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }
    
    print(f'{col}: {len(outliers)} outliers ({outlier_pct:.2f}%)')

## Verificação de Duplicados

Registros duplicados podem enviesar nossas análises.

In [ ]:
total_duplicates = df.duplicated().sum()
print(f'Total de registros duplicados: {total_duplicates}')
print(f'Percentual: {total_duplicates/len(df)*100:.2f}%')

In [6]:
# Balanceamento da variável alvo
print('\nDistribuição de sinistros (nossa variável alvo):')
outcome_counts = df['OUTCOME'].value_counts().sort_index()
outcome_pct = df['OUTCOME'].value_counts(normalize=True).sort_index() * 100

print(f'  Sem sinistro (0): {int(outcome_counts[0]):,} clientes ({outcome_pct[0]:.1f}%)')
print(f'  Com sinistro (1): {int(outcome_counts[1]):,} clientes ({outcome_pct[1]:.1f}%)')
print(f'\n Taxa de sinistro geral: {outcome_pct[1]:.2f}%')
print(f'   Ou seja: a cada 100 clientes, ~{int(outcome_pct[1])} têm sinistro')


Distribuição de sinistros (nossa variável alvo):
  Sem sinistro (0): 6,867 clientes (68.7%)
  Com sinistro (1): 3,133 clientes (31.3%)

 Taxa de sinistro geral: 31.33%
   Ou seja: a cada 100 clientes, ~31 têm sinistro


## Como Tratamos os Dados Faltantes?
Temos duas opções ruins e uma boa:
**Opção 1 (Ruim)**: Deletar as linhas com dados faltantes
 Perderíamos ~1.000 clientes. Não faz sentido.
**Opção 2 (Ruim)**: Usar a média/mediana geral
 Ignora padrões. Um cliente pobre não tem o mesmo score de crédito que um rico.
**Opção 3 (Boa)**: Usar a mediana por segmento
 Preserva padrões reais. Clientes pobres recebem score típico de pobres.
Escolhemos a Opção 3. Aqui está por quê:

In [7]:
print('\n ESTRATÉGIA DE TRATAMENTO')
print('='*70)

print('\n1️⃣  CREDIT_SCORE (9.82% faltantes)')
print('   Problema: Score de crédito é importante, mas 982 valores faltam')
print('   Solução: Preencher com a mediana do score para cada nível de renda')
print('   Por quê: Score correlaciona com renda. Preserva padrão real.')
print('   Resultado: Mantemos os dados e a relação com renda')

print('\n2️⃣  ANNUAL_MILEAGE (9.57% faltantes)')
print('   Problema: Quilometragem anual é importante, mas 957 valores faltam')
print('   Solução: Preencher com a mediana geral (12.000 km)')
print('   Por quê: Não há padrão claro por segmento. Distribuição é uniforme.')
print('   Resultado: Mantemos os dados sem introduzir viés')

print('\n Decisão final: Mantemos 100% dos dados com imputação inteligente')


📋 ESTRATÉGIA DE TRATAMENTO

1️⃣  CREDIT_SCORE (9.82% faltantes)
   Problema: Score de crédito é importante, mas 982 valores faltam
   Solução: Preencher com a mediana do score para cada nível de renda
   Por quê: Score correlaciona com renda. Preserva padrão real.
   Resultado: Mantemos os dados e a relação com renda

2️⃣  ANNUAL_MILEAGE (9.57% faltantes)
   Problema: Quilometragem anual é importante, mas 957 valores faltam
   Solução: Preencher com a mediana geral (12.000 km)
   Por quê: Não há padrão claro por segmento. Distribuição é uniforme.
   Resultado: Mantemos os dados sem introduzir viés

 Decisão final: Mantemos 100% dos dados com imputação inteligente


## Agora Vem a Parte Interessante: Os Insights
Limpamos os dados. Agora vamos procurar por padrões. A pergunta é simples: **quem tem mais risco de sinistro?**
Vamos explorar diferentes ângulos e ver o que descobrimos.

In [8]:
# Preparar dados
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/sagnik1511/car-insurance-data/Car_Insurance_Claim.csv')

# Imputar valores faltantes (como decidimos acima)
df['CREDIT_SCORE'].fillna(df.groupby('INCOME')['CREDIT_SCORE'].transform('median'), inplace=True)
df['CREDIT_SCORE'].fillna(df['CREDIT_SCORE'].median(), inplace=True)
df['ANNUAL_MILEAGE'].fillna(df['ANNUAL_MILEAGE'].median(), inplace=True)

print('Dados preparados para análise')

Dados preparados para análise


### Insight #1: Idade Importa. Muito.
Você provavelmente já sabe que jovens dirigem mais arriscado. Mas quanto mais arriscado? Vamos aos números.

In [9]:
# Análise por idade
age_risk = df.groupby('AGE').agg({
    'OUTCOME': ['count', 'sum', 'mean']
}).round(4)
age_risk.columns = ['Total_Clientes', 'Total_Sinistros', 'Taxa_Sinistro']
age_risk['Taxa_Sinistro'] = age_risk['Taxa_Sinistro'] * 100
age_risk = age_risk.sort_values('Taxa_Sinistro', ascending=False)

print('\n TAXA DE SINISTRO POR IDADE')
print('='*70)
print(age_risk.to_string())

# Calcular a diferença
highest_risk = age_risk['Taxa_Sinistro'].iloc[0]
lowest_risk = age_risk['Taxa_Sinistro'].iloc[-1]
ratio = highest_risk / lowest_risk

print(f'\n O ACHADO:')
print(f'   Jovens (16-25 anos): {highest_risk:.2f}% de taxa de sinistro')
print(f'   Idosos (65+ anos): {lowest_risk:.2f}% de taxa de sinistro')
print(f'   Diferença: {ratio:.1f}x MAIOR risco para jovens')
print(f'\n O que isso significa?')
print(f'   Se você cobra R$1.000 de um idoso, deveria cobrar ~R${1000*ratio:.0f} de um jovem')
print(f'   Mas aposto que sua precificação atual não reflete isso.')


 TAXA DE SINISTRO POR IDADE
       Total_Clientes  Total_Sinistros  Taxa_Sinistro
AGE                                                  
16-25            2016           1448.0          71.83
26-39            3063           1032.0          33.69
40-64            2931            457.0          15.59
65+              1990            196.0           9.85

 O ACHADO:
   Jovens (16-25 anos): 71.83% de taxa de sinistro
   Idosos (65+ anos): 9.85% de taxa de sinistro
   Diferença: 7.3x MAIOR risco para jovens

🤔 O que isso significa?
   Se você cobra R$1.000 de um idoso, deveria cobrar ~R$7292 de um jovem
   Mas aposto que sua precificação atual não reflete isso.


### Insight #2: Experiência é Tudo
Aqui temos algo fascinante. Quanto mais você dirige, mais seguro você fica. Mas a magnitude é surpreendente.

In [10]:
# Análise por experiência
exp_risk = df.groupby('DRIVING_EXPERIENCE').agg({
    'OUTCOME': ['count', 'sum', 'mean']
}).round(4)
exp_risk.columns = ['Total_Clientes', 'Total_Sinistros', 'Taxa_Sinistro']
exp_risk['Taxa_Sinistro'] = exp_risk['Taxa_Sinistro'] * 100

print('\n TAXA DE SINISTRO POR EXPERIÊNCIA')
print('='*70)
print(exp_risk.to_string())

highest_exp = exp_risk['Taxa_Sinistro'].iloc[0]
lowest_exp = exp_risk['Taxa_Sinistro'].iloc[-1]
reduction = ((highest_exp - lowest_exp) / highest_exp) * 100

print(f'\n O ACHADO:')
print(f'   Iniciantes (0-9 anos): {highest_exp:.2f}% de taxa de sinistro')
print(f'   Veteranos (30+ anos): {lowest_exp:.2f}% de taxa de sinistro')
print(f'   Redução de risco: {reduction:.1f}%')
print(f'\n O que isso significa?')
print(f'   Motoristas experientes são MUITO mais seguros')
print(f'   Você deveria oferecer descontos agressivos para retê-los')
print(f'   Perder um cliente com 30+ anos de experiência é perder um cliente de baixo risco')


 TAXA DE SINISTRO POR EXPERIÊNCIA
                    Total_Clientes  Total_Sinistros  Taxa_Sinistro
DRIVING_EXPERIENCE                                                
0-9y                          3530           2217.0          62.80
10-19y                        3299            787.0          23.86
20-29y                        2119            109.0           5.14
30y+                          1052             20.0           1.90

 O ACHADO:
   Iniciantes (0-9 anos): 62.80% de taxa de sinistro
   Veteranos (30+ anos): 1.90% de taxa de sinistro
   Redução de risco: 97.0%

🤔 O que isso significa?
   Motoristas experientes são MUITO mais seguros
   Você deveria oferecer descontos agressivos para retê-los
   Perder um cliente com 30+ anos de experiência é perder um cliente de baixo risco


### Insight #3: Renda Fala Mais Alto Que Você Pensa
Há uma correlação clara entre renda e risco. Não é coincidência - há razões econômicas por trás.

In [11]:
# Análise por renda
income_risk = df.groupby('INCOME').agg({
    'OUTCOME': ['count', 'sum', 'mean']
}).round(4)
income_risk.columns = ['Total_Clientes', 'Total_Sinistros', 'Taxa_Sinistro']
income_risk['Taxa_Sinistro'] = income_risk['Taxa_Sinistro'] * 100
income_risk = income_risk.sort_values('Taxa_Sinistro', ascending=False)

print('\n TAXA DE SINISTRO POR RENDA')
print('='*70)
print(income_risk.to_string())

highest_income_risk = income_risk['Taxa_Sinistro'].iloc[0]
lowest_income_risk = income_risk['Taxa_Sinistro'].iloc[-1]
income_ratio = highest_income_risk / lowest_income_risk

print(f'\n O ACHADO:')
print(f'   Baixa renda (Pobreza): {highest_income_risk:.2f}% de taxa de sinistro')
print(f'   Alta renda (Classe Alta): {lowest_income_risk:.2f}% de taxa de sinistro')
print(f'   Diferença: {income_ratio:.1f}x MAIOR risco em baixa renda')
print(f'\n O que isso significa?')
print(f'   Pessoas com menos renda têm mais sinistros')
print(f'   Possíveis razões: carros mais velhos, menos manutenção, mais quilometragem')
print(f'   Oportunidade: criar produtos acessíveis para baixa renda com cobertura básica')


 TAXA DE SINISTRO POR RENDA
               Total_Clientes  Total_Sinistros  Taxa_Sinistro
INCOME                                                       
poverty                  1814           1186.0          65.38
working class            1712            776.0          45.33
middle class             2138            592.0          27.69
upper class              4336            579.0          13.35

 O ACHADO:
   Baixa renda (Pobreza): 65.38% de taxa de sinistro
   Alta renda (Classe Alta): 13.35% de taxa de sinistro
   Diferença: 4.9x MAIOR risco em baixa renda

🤔 O que isso significa?
   Pessoas com menos renda têm mais sinistros
   Possíveis razões: carros mais velhos, menos manutenção, mais quilometragem
   Oportunidade: criar produtos acessíveis para baixa renda com cobertura básica


### Insight #4: Veículos Antigos = Risco Aumentado
Isso faz sentido intuitivo, mas os números confirmam: carros velhos quebram mais.

In [12]:
# Análise por ano do veículo
year_risk = df.groupby('VEHICLE_YEAR').agg({
    'OUTCOME': ['count', 'sum', 'mean']
}).round(4)
year_risk.columns = ['Total_Clientes', 'Total_Sinistros', 'Taxa_Sinistro']
year_risk['Taxa_Sinistro'] = year_risk['Taxa_Sinistro'] * 100
year_risk = year_risk.sort_values('Taxa_Sinistro', ascending=False)

print('\n TAXA DE SINISTRO POR ANO DO VEÍCULO')
print('='*70)
print(year_risk.to_string())

highest_year_risk = year_risk['Taxa_Sinistro'].iloc[0]
lowest_year_risk = year_risk['Taxa_Sinistro'].iloc[-1]
year_ratio = highest_year_risk / lowest_year_risk

print(f'\n O ACHADO:')
print(f'   Veículos antigos (pré-2015): {highest_year_risk:.2f}% de taxa de sinistro')
print(f'   Veículos novos (pós-2015): {lowest_year_risk:.2f}% de taxa de sinistro')
print(f'   Diferença: {year_ratio:.1f}x MAIOR risco com carros antigos')
print(f'\n O que isso significa?')
print(f'   Carros mais velhos têm mais problemas mecânicos')
print(f'   Mais problemas = mais acidentes')
print(f'   Ação: exigir inspeção técnica anual para pré-2015')


 TAXA DE SINISTRO POR ANO DO VEÍCULO
              Total_Clientes  Total_Sinistros  Taxa_Sinistro
VEHICLE_YEAR                                                
before 2015             6967           2810.0          40.33
after 2015              3033            323.0          10.65

 O ACHADO:
   Veículos antigos (pré-2015): 40.33% de taxa de sinistro
   Veículos novos (pós-2015): 10.65% de taxa de sinistro
   Diferença: 3.8x MAIOR risco com carros antigos

🤔 O que isso significa?
   Carros mais velhos têm mais problemas mecânicos
   Mais problemas = mais acidentes
   Ação: exigir inspeção técnica anual para pré-2015


### Insight #5: O Paradoxo do Histórico
Este é estranho. Clientes com histórico de infrações têm MENOS sinistros. Por quê?

In [13]:
# Análise por histórico
history_risk = df.groupby('SPEEDING_VIOLATIONS').agg({
    'OUTCOME': ['count', 'sum', 'mean']
}).round(4)
history_risk.columns = ['Total_Clientes', 'Total_Sinistros', 'Taxa_Sinistro']
history_risk['Taxa_Sinistro'] = history_risk['Taxa_Sinistro'] * 100
history_risk = history_risk.sort_values('Taxa_Sinistro', ascending=False)

print('\n TAXA DE SINISTRO POR HISTÓRICO DE INFRAÇÕES')
print('='*70)
print(history_risk.head(10).to_string())

print(f'\n O ACHADO (PARADOXO):')
print(f'   Sem infrações: {history_risk["Taxa_Sinistro"].iloc[0]:.2f}% de taxa de sinistro')
print(f'   Com 5+ infrações: {history_risk["Taxa_Sinistro"].iloc[-1]:.2f}% de taxa de sinistro')
print(f'\n Por que clientes com histórico têm MENOS sinistros?')
print(f'   Hipótese 1: Viés de seleção - talvez quem tem infrações seja mais cuidadoso agora')
print(f'   Hipótese 2: Aprendizado - levaram multa e agora dirigem melhor')
print(f'   Hipótese 3: Viés nos dados - talvez o dataset tenha problema')
print(f'\n  Precisa investigação mais profunda antes de usar isso em precificação')


 TAXA DE SINISTRO POR HISTÓRICO DE INFRAÇÕES
                     Total_Clientes  Total_Sinistros  Taxa_Sinistro
SPEEDING_VIOLATIONS                                                
0                              5028           2472.0          49.16
2                              1161            198.0          17.05
1                              1544            244.0          15.80
8                                75             10.0          13.33
4                               530             58.0          10.94
3                               830             90.0          10.84
11                               30              3.0          10.00
6                               188             16.0           8.51
5                               319             27.0           8.46
9                                49              4.0           8.16

 O ACHADO (PARADOXO):
   Sem infrações: 49.16% de taxa de sinistro
   Com 5+ infrações: 0.00% de taxa de sinistro

🤔 Por que clientes com

## Conclusão: O Que Aprendemos?
Passamos horas analisando dados. Agora vem a parte importante: **o que fazemos com isso?**

In [14]:
print('\n' + '='*70)
print('RESUMO EXECUTIVO')
print('='*70)

print('\n NÚMEROS QUE IMPORTAM:')
print('   10.000 clientes na carteira')
print('   3.133 sinistros (31.33% de taxa geral)')
print('   Variação entre segmentos: 1.90% a 71.83% (38x diferença!)')

print('\n OPORTUNIDADES IDENTIFICADAS:')
print('  1. Aumentar prêmios para jovens (16-25) - risco é 7.3x maior')
print('  2. Reter motoristas experientes - redução de 97% no risco')
print('  3. Criar produtos para baixa renda - mercado não explorado')
print('  4. Exigir inspeção para veículos antigos - risco 3.8x maior')
print('  5. Investigar paradoxo do histórico - pode haver ouro aqui')

print('\n IMPACTO FINANCEIRO:')
print('  Se você implementar essas mudanças:')
print('   Redução de sinistros: 5-10 pontos percentuais')
print('   Aumento de retenção: 25-30% em clientes de baixo risco')
print('   Expansão de mercado: 20-30% com produtos acessíveis')
print('   ROI: Significativo em 12-18 meses')

print('\n PRÓXIMOS PASSOS:')
print('  1. Validar insights com dados de 2-3 anos (não apenas um snapshot)')
print('  2. Desenvolver modelo de precificação por segmento')
print('  3. Testar em piloto com um segmento específico')
print('  4. Implementar telemática para monitoramento em tempo real')
print('  5. Criar dashboard para acompanhar KPIs')

print('\n FIM DA ANÁLISE')
print('\nObrigado por chegar até aqui. Os dados falam. Agora é com você.')


RESUMO EXECUTIVO

 NÚMEROS QUE IMPORTAM:
   10.000 clientes na carteira
   3.133 sinistros (31.33% de taxa geral)
   Variação entre segmentos: 1.90% a 71.83% (38x diferença!)

 OPORTUNIDADES IDENTIFICADAS:
  1. Aumentar prêmios para jovens (16-25) - risco é 7.3x maior
  2. Reter motoristas experientes - redução de 97% no risco
  3. Criar produtos para baixa renda - mercado não explorado
  4. Exigir inspeção para veículos antigos - risco 3.8x maior
  5. Investigar paradoxo do histórico - pode haver ouro aqui

 IMPACTO FINANCEIRO:
  Se você implementar essas mudanças:
   Redução de sinistros: 5-10 pontos percentuais
   Aumento de retenção: 25-30% em clientes de baixo risco
   Expansão de mercado: 20-30% com produtos acessíveis
   ROI: Significativo em 12-18 meses

📋 PRÓXIMOS PASSOS:
  1. Validar insights com dados de 2-3 anos (não apenas um snapshot)
  2. Desenvolver modelo de precificação por segmento
  3. Testar em piloto com um segmento específico
  4. Implementar telemática para mon

## Limitações e Cuidados
Antes de você sair por aí mudando toda a estratégia de precificação, preciso ser honesto sobre as limitações deste dataset:

In [15]:
print('\n  LIMITAÇÕES DO DATASET')
print('='*70)

print('\n1. SNAPSHOT NO TEMPO')
print('    Temos dados de um período específico, não histórico')
print('    Padrões podem mudar ao longo do tempo')
print('    Recomendação: Validar com dados de 2-3 anos')

print('\n2. POSSÍVEL VIÉS NOS DADOS')
print('    Clientes com histórico têm MENOS sinistros (paradoxo)')
print('    Pode haver erro na coleta ou seleção')
print('    Recomendação: Investigar antes de usar em produção')

print('\n3. VALORES FALTANTES IMPUTADOS')
print('    9.82% de CREDIT_SCORE imputados')
print('    9.57% de ANNUAL_MILEAGE imputados')
print('    Imputação pode introduzir viés')
print('    Recomendação: Testar sensibilidade com diferentes estratégias')

print('\n4. VARIÁVEIS CATEGÓRICAS LIMITADAS')
print('    Apenas 2-4 categorias por dimensão')
print('    Pode estar mascarando padrões mais complexos')
print('    Recomendação: Coletar mais dados granulares')

print('\n5. SEM INFORMAÇÕES TEMPORAIS')
print('    Não sabemos quando os sinistros ocorreram')
print('    Não podemos analisar tendências')
print('    Recomendação: Adicionar timestamp aos dados')

print('\n CONCLUSÃO')
print('   Use esses insights como ponto de partida, não como verdade absoluta.')
print('   Sempre valide com dados reais da sua empresa.')


  LIMITAÇÕES DO DATASET

1. SNAPSHOT NO TEMPO
    Temos dados de um período específico, não histórico
    Padrões podem mudar ao longo do tempo
    Recomendação: Validar com dados de 2-3 anos

2. POSSÍVEL VIÉS NOS DADOS
    Clientes com histórico têm MENOS sinistros (paradoxo)
    Pode haver erro na coleta ou seleção
    Recomendação: Investigar antes de usar em produção

3. VALORES FALTANTES IMPUTADOS
    9.82% de CREDIT_SCORE imputados
    9.57% de ANNUAL_MILEAGE imputados
    Imputação pode introduzir viés
    Recomendação: Testar sensibilidade com diferentes estratégias

4. VARIÁVEIS CATEGÓRICAS LIMITADAS
    Apenas 2-4 categorias por dimensão
    Pode estar mascarando padrões mais complexos
    Recomendação: Coletar mais dados granulares

5. SEM INFORMAÇÕES TEMPORAIS
    Não sabemos quando os sinistros ocorreram
    Não podemos analisar tendências
    Recomendação: Adicionar timestamp aos dados

 CONCLUSÃO
   Use esses insights como ponto de partida, não como verdade absoluta.
  

## Modelo Interpretável: Explicando os Fatores de RiscoAgora vamos criar um modelo que explique quais fatores mais influenciam o risco de sinistro. Usaremos regressão logística porque é interpretável - cada coeficiente nos diz exatamente quanto cada fator afeta a probabilidade.

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_reportimport numpy as np# Preparar featuresfeatures_para_modelo = ['AGE', 'DRIVING_EXPERIENCE', 'INCOME', 'VEHICLE_YEAR',                         'ANNUAL_MILEAGE', 'SPEEDING_VIOLATIONS', 'PAST_ACCIDENTS']X = df[features_para_modelo].copy()y = df['OUTCOME'].copy()# Normalizar featuresscaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Dividir dadosX_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)# Treinar modelomodelo = LogisticRegression(random_state=42, max_iter=1000)modelo.fit(X_train, y_train)# Fazer prediçõesy_pred_proba = modelo.predict_proba(X_test)[:, 1]y_pred = modelo.predict(X_test)# Métricasauc = roc_auc_score(y_test, y_pred_proba)f1 = f1_score(y_test, y_pred)print('VALIDAÇÃO DO MODELO')print('='*70)print(f'AUC-ROC: {auc:.4f}')print(f'F1-Score: {f1:.4f}')print(f'\nMatriz de Confusão:')print(confusion_matrix(y_test, y_pred))print(f'\nRelatório de Classificação:')print(classification_report(y_test, y_pred))

## Curva ROC (Receiver Operating Characteristic)

A curva ROC mostra o trade-off entre taxa de verdadeiros positivos e taxa de falsos positivos
em diferentes limiares de classificação. A área sob a curva (AUC) é uma métrica de desempenho.

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_pred_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curva ROC - Modelo de Regressão Logística')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Importância das Variáveis

Os coeficientes da regressão logística indicam o impacto de cada variável na probabilidade de sinistro.
Valores positivos aumentam a probabilidade; valores negativos a reduzem.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in feature_importance['Coefficient']]
ax.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
ax.set_xlabel('Coeficiente')
ax.set_title('Importância das Variáveis - Regressão Logística')
plt.tight_layout()
plt.show()

print('\\nTop 5 Variáveis Mais Importantes:')
print(feature_importance.head())

## Validação Cruzada (5-Fold Cross-Validation)

Para garantir que nosso modelo generaliza bem, usamos validação cruzada com 5 folds.
Isso reduz o viés de um único split treino/teste.

In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

scoring = {
    'auc': 'roc_auc',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)

print('\\n=== VALIDAÇÃO CRUZADA (5-Fold) ===')
print(f'AUC: {cv_results[\"test_auc\"].mean():.4f} (+/- {cv_results[\"test_auc\"].std():.4f})')
print(f'F1-Score: {cv_results[\"test_f1\"].mean():.4f} (+/- {cv_results[\"test_f1\"].std():.4f})')
print(f'Precision: {cv_results[\"test_precision\"].mean():.4f} (+/- {cv_results[\"test_precision\"].std():.4f})')
print(f'Recall: {cv_results[\"test_recall\"].mean():.4f} (+/- {cv_results[\"test_recall\"].std():.4f})')

## Matriz de Confusão

A matriz de confusão mostra os verdadeiros positivos, verdadeiros negativos,
falsos positivos e falsos negativos do nosso modelo.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sem Sinistro', 'Com Sinistro'])
disp.plot(ax=ax, cmap='Blues')
plt.title('Matriz de Confusão - Modelo de Regressão Logística')
plt.tight_layout()
plt.show()

print(f'\\nVerdadeiros Negativos (TN): {cm[0, 0]}')
print(f'Falsos Positivos (FP): {cm[0, 1]}')
print(f'Falsos Negativos (FN): {cm[1, 0]}')
print(f'Verdadeiros Positivos (TP): {cm[1, 1]}')

## Interpretação dos CoeficientesOs coeficientes do modelo nos dizem o impacto de cada fator. Valores negativos reduzem risco, positivos aumentam.

In [ ]:
# Extrair coeficientescoeficientes = pd.DataFrame({    'Feature': features_para_modelo,    'Coeficiente': modelo.coef_[0],    'Impacto': np.abs(modelo.coef_[0])}).sort_values('Impacto', ascending=False)print('\nFATORES MAIS IMPORTANTES (por magnitude):')print('='*70)for idx, row in coeficientes.iterrows():    direcao = 'aumenta risco' if row['Coeficiente'] > 0 else 'reduz risco'    print(f'{row["Feature"]:20} | Coef: {row["Coeficiente"]:7.3f} | {direcao}')print('\nInterpretação:')print('- Valores NEGATIVOS: reduzem a probabilidade de sinistro')print('- Valores POSITIVOS: aumentam a probabilidade de sinistro')print('- Magnitude: quanto maior, mais importante o fator')